# Project Title: Flickr8k Image Captioning with CNN-LSTM Architecture

**Description:** An encoder-decoder model utilizing a pre-trained VGG16 for image feature extraction and a PyTorch LSTM network for sequence generation.

**Dataset:** Flickr8k (via Kaggle); 8,000 images with 5 captions each.

**Objective:** Generate human-readable captions and achieve measurable BLEU-1 to BLEU-4 scores.

# 1. Environment Setup & Configuration

This section initializes the project environment, loads **dependencies**, and configures the **hardware acceleration** required to train the **CNN-LSTM architecture** efficiently.

## 1.1 Library Imports

We import standard Python libraries for file and string manipulation, alongside third-party libraries such as **pandas** and **numpy** for data handling. We also import **torch** and **torchvision** modules, which form the core framework for our Deep Learning models, datasets, and image transformations.

## 1.2 Global Constants

We define our model **hyperparameters** and dataset paths as uppercase **global constants**. Centralizing **BATCH_SIZE**, **LEARNING_RATE**, and **EPOCHS** allows for rapid experimentation, while setting a **RANDOM_SEED** ensures our training splits and weight initializations remain reproducible across runs.

## 1.3 Hardware Accelerator

We implement **device-agnostic code** by checking for `cuda` (NVIDIA GPUs) or `mps` (Apple Silicon). If a compatible accelerator is found, the **DEVICE** constant routes all tensor computations to the **GPU**, drastically reducing the training time for the **VGG16 feature extraction** and **LSTM sequence generation**.

# 2. Data Engineering & Preprocessing

## 2.1 Data Ingestion

We parse the raw text file containing all captions (Flickr8k.token.txt), isolating the **image_id** from its corresponding **caption string**. We load this directly into a **pandas DataFrame** (`captions_df`) for structured manipulation.

## 2.2 Exploratory Data Analysis (EDA)

While comprehensive visual **EDA** would occur in a notebook cell (plotting caption length distributions), at the script level, we prepare the data structure to easily extract metadata, such as **max caption length** and **vocabulary size**, which directly inform our embedding layer dimensions.

## 2.3 Feature Engineering

We clean the text by converting it to **lowercase**, stripping **punctuation**, and filtering out **single-character tokens**. We append `<start>` and `<end>` tokens to demarcate sequence boundaries. Additionally, we implement a **VocabularyBuilder** to handle frequency-based **tokenization**, converting our text sequences into **numerical indices**.

## 2.4 Dataset Splitting

Flickr8k provides explicit text files listing the **image IDs** for **training**, **validation (dev)**, and **testing splits**. We extract these IDs into sets and filter our primary `captions_df` to create distinct `train_df`, `val_df`, and `test_df` partitions, preventing **data leakage**.

# EDA Insights Summary

## 1. Data Quality & Diversity (From Visual Inspection)

*   **Insight:** Each image correctly maps to **5 distinct captions**. These captions describe the exact same scene but vary in **focus**, **vocabulary**, and **sentence structure** (e.g., focusing on the action vs. the clothing).

*   **Actionable Takeaway:** This variance is excellent for deep learning. It will prevent the model from memorizing a single strict phrasing and instead teach it to generalize the mapping of **visual features** to **semantic meaning**.

## 2. Sequence Length Optimization (From Length Distribution)

*   **Insight:** While the absolute longest caption is **38 words**, the distribution is heavily right-skewed. The **mean length** is ~12 words, and **95% of all captions are 19 words or shorter**.

*   **Actionable Takeaway:** We will set our **MAX_LEN hyperparameter to 19** (plus 2 for our `<start>` and `<end>` tokens, making it 21). If we padded every sequence to 38, we would waste roughly **50% of our memory and LSTM compute cycles** on empty `<pad>` tokens. Truncating the top 5% of exceptionally long sentences is a standard trade-off for massive gains in training efficiency.

## 3. Vocabulary Distribution & Cleaning Needs (From Word Frequencies)

*   **Insight:** The top frequent words are dominated by articles/prepositions ("a", "in", "the", "on") and common subjects ("dog", "man", "boy", "white", "black"). This indicates the dataset is heavily biased toward outdoor scenes involving people and animals.

*   **Actionable Takeaway:** Notice that the period (.) is the second most frequent "word." This proves exactly why our Section 2.3 (**Feature Engineering**) step is critical. We must **strip punctuation**, **lower-case everything**, and **remove single-character words** (except maybe "a") so our embedding layer doesn't waste space learning useless characters.

# 3. PyTorch Data Pipelines

## 3.1 Custom Dataset Class
We define a **PyTorch Dataset** that takes our DataFrames, reads the corresponding image from the disk, applies computer vision **transformations** (resizing and normalizing for VGG16), and converts the cleaned text into a **numerical tensor** using our VocabularyBuilder.

## 3.2 DataLoader Instantiation
Because captions have different lengths, we cannot stack them into a perfectly rectangular batch tensor by default. We create a custom **CollateFn** to append `<pad>` tokens (zeros) to the shorter sentences in a batch, ensuring uniform tensor dimensions before feeding them to the **DataLoader**.

# 4. Model Architecture

## 4.1 Class Definition
We are building an **Encoder-Decoder architecture**:

**Encoder (CNN)**: We use a **pre-trained VGG16 model**. To save memory and training time, we **freeze the core convolutional layers** (so their weights don't update) and replace the final classification layer. Instead of classifying 1,000 ImageNet categories, our modified VGG16 will output a **dense feature vector** (e.g., size 256) that summarizes the visual content of the image.

**Decoder (RNN)**: We build an **LSTM network**. It takes the **image feature vector** as the "first word" in a sequence, followed by the **embedded text tokens**. The LSTM learns to predict the next token in the sequence based on the image context and all previously seen words.

## 4.2 Model Initialization
We instantiate both the **Encoder** and **Decoder**, pass in our **hyperparameter dimensions** (**EMBED_SIZE**, **HIDDEN_SIZE**, **VOCAB_SIZE**), and move them to our configured **DEVICE**.

## 4.3 Summary & Sanity Check
Before writing the complex training loop, we pass our `sample_imgs` and `sample_caps` through the initialized models to ensure the mathematical **matrix multiplications** line up perfectly and output the correct final shape.

# 5. Training Logic & Optimization

## 5.1 Loss & Optimizer
We define **CrossEntropyLoss** as our objective function, explicitly configuring it to ignore the `<pad>` token (**PAD_IDX**). This prevents the model from artificially reducing its loss by easily predicting sequences of padded zeros. We instantiate the **Adam optimizer**, passing it the trainable parameters of the decoder and the projection embedding layer of the frozen CNN encoder.

## 5.2 Training Step Function
We construct `execute_train_step()` to handle the standard PyTorch training operations: zeroing gradients, performing the forward pass through both models, flattening the spatial and sequence dimensions to calculate loss across the batch, running backpropagation, and updating weights.

## 5.3 Validation Step Function
We construct `execute_val_step()` utilizing `torch.no_grad()` to freeze weight tracking and `model.eval()` to disable dropout layers. This efficiently calculates the validation loss without consuming unnecessary GPU memory, allowing us to monitor for overfitting during the epoch loop.

# 9. Advanced Model Evolution: Spatial Attention & Beam Search
The primary limitation of our current model is that the Encoder compresses the entire image into a single vector, forcing the Decoder to "remember" the whole scene at once. In this improved approach, we allow the Decoder to "look" at specific regions of the image for each word it generates.

## 9.1 Attention-Based Encoder
Instead of using the final fully-connected layer of VGG16, we extract the **Convolutional Feature Maps** (e.g., the output of the last Conv layer, typically $7 \times 7 \times 512$). This provides **49 individual spatial "grid cells,"** each representing a specific area of the image.

## 9.2 Bahdanau (Additive) Attention Layer
We implement an **Attention class** that calculates "alignment scores" between the current LSTM hidden state and the 49 spatial grid cells.
*   **Context Vector:** A weighted sum of the grid cells based on these scores.
*   **Focus:** If the model is predicting the word "water," the attention weights for the grid cells containing blue/liquid textures will be higher.

## 9.3 Implementing Beam Search Decoding
To solve the "young young" repetition loops found in the baby photo, we replace **Greedy Search** with **Beam Search**.
*   Instead of picking the top 1 word, we keep the top $k$ (e.g., $k=3$ or $5$) most likely sequences.
*   At each step, we expand all $k$ sequences and keep the $k$ best total paths.
*   **Result:** This leads to more grammatically stable sentences and higher BLEU-4 scores because the model considers the long-term impact of its word choices.

## 9.4 The Attention-Decoder Integration
In this step, we build the **DecoderWithAttention** class. Unlike our baseline, this decoder recalculates the **Context Vector** at every single LSTM time step.

### 9.4.1 Decoder Class Definition
The decoder combines the **word embedding** of the previous word with the current **visual context vector** before feeding them into the **LSTM**.

### 9.4.2 Advanced Training Setup
Because the Attention architecture is more complex, we typically use a **Learning Rate Scheduler** to reduce the **LR** when the validation loss plateaus.

## 9.5 Advanced Training & Attention Visualization
### 9.5.1 Advanced Training Loop
We will run a short training session. Because the **Attention model** is more "opinionated" about where to look, you will likely see the **validation loss** drop faster than the baseline.

### 9.5.2 Visualizing Attention (The "Heatmap")
This function takes a test image, generates a caption, and for every word generated, it produces a **$7 \times 7$ heatmap** showing the **attention weights** ($\alpha$). This is the best evidence for your "Analysis" section.

## 2.3 Feature Engineering
In this step, we clean the **raw text** (lowercasing, removing punctuation, filtering out single characters) and append our **sequence boundaries** (`<start>` and `<end>`). We then build a **VocabularyBuilder** to map these standardized tokens to **numerical indices**, filtering out rare words to optimize the **embedding matrix**.

## 2.4 Dataset Splitting
We map the provided text files (trainImages.txt, devImages.txt, testImages.txt) against our **main DataFrame** to create **isolated datasets**. Crucially, we implement a **sanity check** to verify **zero data leakage** between our splits.

# 6. Training Execution (The Loop)

## 6.1 The Epoch Loop
We will iterate through our specified number of **epochs**. In each epoch, we first run the **training step** to update the weights, followed immediately by the **validation step** to evaluate how well the model generalizes to unseen images.

## 6.2 Progress Monitoring & Checkpointing
Deep learning models are prone to overfitting. To safeguard our progress, we implement **Model Checkpointing**. We track the `val_loss` at the end of every epoch. If it drops to a new historical low, we save the model's **weights** (`state_dict`) to the disk. We also record the **loss history** so we can plot learning curves later.

# 7. Evaluation & Error Analysis
To score our model, we must write an **Inference Pipeline** (to generate a caption for a brand-new image) and an **Evaluation Loop** (to calculate the **BLEU-1** through **BLEU-4** scores against the 5 ground-truth human captions).

## 7.1 Inference / Decoding Logic (Greedy Search)
We write a function that takes a single image, extracts its features using the **VGG16 encoder**, and feeds the `<start>` token to the **LSTM**. The LSTM predicts the next word, which we append to our sequence, feeding it back into the LSTM until it outputs the `<end>` token.

## 7.2 BLEU Score Calculation
The **BLEU (Bilingual Evaluation Understudy) metric** measures how many words and phrases (n-grams) in our predicted caption match the human reference captions.

*   **BLEU-1:** Measures individual word matches.
*   **BLEU-2:** Measures 2-word phrase matches (bigrams).
*   **BLEU-3:** Measures 3-word phrase matches (trigrams).
*   **BLEU-4:** Measures 4-word phrase matches.

# 8. Deployment & Inference

## 8.1 Model Export
We save the **state dictionaries** of both the **Encoder** and **Decoder**. This allows you to reload the model later for inference without needing to spend another 40 minutes training.

## 8.2 Inference Pipeline & Visualization
We implement a function that selects random images from the `test_df`, processes them through the **vision-to-language pipeline**, and displays the image alongside the **Generated Caption** and the **Human Reference**.

# 9. Advanced Model Evolution: Spatial Attention & Beam Search
The primary limitation of our current model is that the Encoder compresses the entire image into a single vector, forcing the Decoder to "remember" the whole scene at once. In this improved approach, we allow the Decoder to "look" at specific regions of the image for each word it generates.

## 9.1 Attention-Based Encoder
Instead of using the final fully-connected layer of VGG16, we extract the **Convolutional Feature Maps** (e.g., the output of the last Conv layer, typically $7 \times 7 \times 512$). This provides **49 individual spatial "grid cells,"** each representing a specific area of the image.

## 9.2 Bahdanau (Additive) Attention Layer
We implement an **Attention class** that calculates "alignment scores" between the current LSTM hidden state and the 49 spatial grid cells.
*   **Context Vector:** A weighted sum of the grid cells based on these scores.
*   **Focus:** If the model is predicting the word "water," the attention weights for the grid cells containing blue/liquid textures will be higher.

## 9.3 Implementing Beam Search Decoding
To solve the "young young" repetition loops found in the baby photo, we replace **Greedy Search** with **Beam Search**.
*   Instead of picking the top 1 word, we keep the top $k$ (e.g., $k=3$ or $5$) most likely sequences.
*   At each step, we expand all $k$ sequences and keep the $k$ best total paths.
*   **Result:** This leads to more grammatically stable sentences and higher BLEU-4 scores because the model considers the long-term impact of its word choices.

## 9.4 The Attention-Decoder Integration
In this step, we build the **DecoderWithAttention** class. Unlike our baseline, this decoder recalculates the **Context Vector** at every single LSTM time step.

### 9.4.1 Decoder Class Definition
The decoder combines the **word embedding** of the previous word with the current **visual context vector** before feeding them into the **LSTM**.

### 9.4.2 Advanced Training Setup
Because the Attention architecture is more complex, we typically use a **Learning Rate Scheduler** to reduce the **LR** when the validation loss plateaus.

## 9.5 Advanced Training & Attention Visualization
### 9.5.1 Advanced Training Loop
In the **Attention-based Decoder**, we predict the next word for each timestep, so we must align the targets by shifting the captions by one (excluding the `<start>` token from the target and the `<end>` token from the input). We will create specific `execute_train_step_att` and `execute_val_step_att` functions to handle the attention-based inputs and then run the training loop.

### 9.5.2 Visualizing Attention (The "Heatmap")
This function takes a test image, generates a caption, and for every word generated, it produces a **$7 \times 7$ heatmap** showing the **attention weights** ($\alpha$). This is the best evidence for your "Analysis" section.

## 9.6 Visualizing the "Gaze" of the Model
We will create a function that generates a caption for a test image and, for every word produced, extracts the **$7 \times 7$ alpha weights (attention scores)**. We then overlay these weights as a **heatmap** on the original image.

## 9.7 BLEU Score Benchmarking

We will run the `evaluate_bleu_scores` function using our `adv_encoder` and `adv_decoder`. Because we changed the inference logic for attention, I have adjusted the evaluation function slightly below.

# Summary of Advanced Insights
*   **Significant Quantitative Leap:**
    *   **BLEU-1 (0.5127):** A 13% increase over the baseline. The model is now correctly identifying the primary subject in more than half of its word choices.
    *   **BLEU-4 (0.1008):** This is the most critical metric. By jumping from zero to 10%, the model has moved from "labeling" to "storytelling." It is now capable of producing grammatically structured phrases like "jumping into the water" rather than just disconnected keywords.
*   **Spatial "Interrogation":** As seen in your kite-surfing heatmap, the model is no longer guessing. It "interrogates" the image—looking at the person for the noun and the waves for the action. This **Spatial Intelligence** is why the BLEU scores improved so drastically.
*   **Reduced Bias:** Baseline models often overfit to common phrases (the "young young girl" error). The **Attention model** is more grounded in the specific pixels of the current image, which naturally regularizes the output and reduces repetitive "hallucinations."

# Qualitative Insights for Section 9.6
*   **Semantic Focus:** Notice that when the model generates the word "man", the attention heatmap is concentrated directly on the person. This proves the spatial encoder is functioning correctly.
*   **Color Recognition:** For the word "blue", the attention shifts towards the waves and the upper portions of the rider's gear. Even though the rider's shirt is dark, the model is associating "blue" with the dominant color in those regions.
*   **Contextual Action:** When predicting "jumping" and "water", the attention expands to the splash and the surrounding waves. This shows the model understands that the action "jumping" is related to the interaction between the subject and the environment.
*   **Syntactic Stability:** Unlike the baseline "young young girl girl" error, this advanced model produces a perfectly coherent and non-repetitive sentence: "man in blue shirt is jumping into the water."

# Summary of Advanced Insights
*   **Significant Quantitative Leap:**
    *   **BLEU-1 (0.5127):** A 13% increase over the baseline. The model is now correctly identifying the primary subject in more than half of its word choices.
    *   **BLEU-4 (0.1008):** This is the most critical metric. By jumping from zero to 10%, the model has moved from "labeling" to "storytelling." It is now capable of producing grammatically structured phrases like "jumping into the water" rather than just disconnected keywords.
*   **Spatial "Interrogation":** As seen in your kite-surfing heatmap, the model is no longer guessing. It "interrogates" the image—looking at the person for the noun and the waves for the action. This **Spatial Intelligence** is why the BLEU scores improved so drastically.
*   **Reduced Bias:** Baseline models often overfit to common phrases (the "young young girl" error). The **Attention model** is more grounded in the specific pixels of the current image, which naturally regularizes the output and reduces repetitive "hallucinations."

# Qualitative Analysis of Results

## 1. Successes (Object & Context Recognition)
*   **The Snowboarder (ID: 3214...):** This is a strong success. The model correctly identified the subject ("man snowboarder") and the environment ("snowy on hill"). Even though the BLEU score might be low due to phrasing differences, the semantic meaning is 100% accurate.
*   **The Pool (ID: 2461...):** The model correctly identified "boy" and "pool." This shows the VGG16 encoder is doing a great job at recognizing primary subjects and backgrounds.

## 2. Observed Limitations (The "Why" for your BLEU-4 of 0)
*   **Repetitive Loops (The Baby Photo):** The model predicted "young young girl girl in in pink pink dress."
    *   **Insight:** This is a classic symptom of **Greedy Search** and a lack of a "sequence penalty." Because "young" was the most likely word at step 2, it became even more likely at step 3.
*   **Gender/Subject Confusion (The Cafe Photo):** It predicted "two woman people" for a single man sitting at a table.
    *   **Insight:** This suggests the model is picking up on "table/street" context but defaulting to a high-probability phrase it saw too often in training (like "two people").
*   **Grammatical "Stuttering":** Phrases like "in the on street the" or "boy young in boy pool in" show that while the model knows the words, the **LSTM** hasn't yet mastered English syntax. It's placing prepositions randomly because it hasn't trained long enough to learn the "flow" of a sentence.